# 2026-03-04 TODO

|TODO|내용|
|:---:|:---:|
|[v]|1. GCP BigQuery에서 Polars으로 DataFrame 구축|
|[v]|2. FilePath에서 FileName를 분류하기|
|[v]|3. 파일 이름에서 '_'로 분류한 데이터값 정리하기|
|[v]|4. 만들어진 데이터프레임 병합 처리|
|[v]|5. 만들어진 데이터프레임 BigQuery에 올리기|

## 1. GCP BigQuery에서 Polars으로 DataFrame 구축

In [ ]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
from google.cloud import bigquery
from google.oauth2 import service_account

key_path = '/workspaces/Psychological-counseling-researching/.key/testprojects-453622-d1f78fcce8b7.json'
credentials = service_account.Credentials.from_service_account_file(key_path)

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
print(f"Client created with project: {client.project}")


Client created with project: testprojects-453622


In [3]:
import polars as pl
from google.cloud import bigquery

# Perform a query.
QUERY = open('/workspaces/Psychological-counseling-researching/1-text_data/1_test.sql', 'r', encoding='utf-8').read()

query_job = client.query(QUERY)  # API request
rows = query_job.result()  # Waits for query to finish

raw_df = pl.from_arrow(rows.to_arrow())
display(raw_df)

index,FilePath,Speaker,Content
i64,str,str,str
207,"""gs://testprojects-453622-unzip…","""상담사""","""또 그 부분에도 참 또 그렇죠. 그래요. 일단 선생님 …"
206,"""gs://testprojects-453622-unzip…","""내담자""","""그래서 저는 막 야한 거 막 음란물 그거 즐겨 보는 경…"
205,"""gs://testprojects-453622-unzip…","""상담사""","""물론 그 범죄에 대해서 책임을 가지고 또 마땅히 또 처…"
204,"""gs://testprojects-453622-unzip…","""내담자""","""그렇죠."""
203,"""gs://testprojects-453622-unzip…","""상담사""","""글쎄요. 저는 좀 후자적인 부분에 저도 좀 어느 정도 …"
…,…,…,…
4,"""gs://testprojects-453622-unzip…","""상담사""","""그렇군요. 그래요 저녁은 주로 혼자 드세요 아니면 어머…"
3,"""gs://testprojects-453622-unzip…","""내담자""","""저녁은 아직 안 먹었어요. 끝나고 먹으려고요."""
2,"""gs://testprojects-453622-unzip…","""상담사""","""그래요 저녁은 드셨어요? 몇 시에 저녁 드세요?"""


# 2. FilePath에서 FileName를 분류하기

In [4]:
# FilePath에서 FileName 추출
fileNames_df = raw_df.with_columns(
	pl.col("FilePath")
	.str.split("/")
	.list.last()
	.alias("FileName")
)

display(fileNames_df.select(["FilePath", "FileName"]).head(10))

FilePath,FileName
str,str
"""gs://testprojects-453622-unzip…","""resource_addiction_10_check_A0…"
"""gs://testprojects-453622-unzip…","""resource_addiction_10_check_A0…"
"""gs://testprojects-453622-unzip…","""resource_addiction_10_check_A0…"
"""gs://testprojects-453622-unzip…","""resource_addiction_10_check_A0…"
"""gs://testprojects-453622-unzip…","""resource_addiction_10_check_A0…"
"""gs://testprojects-453622-unzip…","""resource_addiction_10_check_A0…"
"""gs://testprojects-453622-unzip…","""resource_addiction_10_check_A0…"
"""gs://testprojects-453622-unzip…","""resource_addiction_10_check_A0…"
"""gs://testprojects-453622-unzip…","""resource_addiction_10_check_A0…"


## 3. 파일 이름에서 '_'로 분류한 데이터값 정리하기

In [5]:
# FileName을 '_' 기준으로 분해해서 컬럼화
parsed_file_name_df = (
	fileNames_df
	.with_columns(
		pl.col("FileName")
		.str.replace(r"\.[^.]+$", "")   # 확장자 제거
		.str.split("_")
		.alias("parts")
	)
	.with_columns(
		pl.col("parts").list.get(0).alias("prefix"),
		pl.col("parts").list.get(1).alias("topic"),
		pl.col("parts").list.get(2).cast(pl.Int64, strict=False).alias("session_no"),
		pl.col("parts").list.get(3).alias("stage"),
		pl.col("parts").list.get(4).alias("file_code"),
	)
	.drop("parts")
)

# 분해된 파일명 기준 건수 집계
parsed_file_name_counts = (
	parsed_file_name_df
	.group_by(["prefix", "topic", "session_no", "stage", "file_code"])
	.len()
	.rename({"len": "row_count"})
	.sort("row_count", descending=True)
)

display(parsed_file_name_df.select(
	["index", "FileName", "prefix", "topic", "session_no", "stage", "file_code"]
).head(10))
parsed_file_name_df.shape

index,FileName,prefix,topic,session_no,stage,file_code
i64,str,str,str,i64,str,str
207,"""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081"""
206,"""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081"""
205,"""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081"""
204,"""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081"""
203,"""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081"""
202,"""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081"""
201,"""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081"""
200,"""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081"""
199,"""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081"""


(78505, 10)

## 4. 만들어진 데이터프레임 병합 처리

다음 의미입니다 (`merged_summary_df` 기준):

|**`row_count`**  
	`prefix, topic, session_no, stage, file_code, Speaker` 조합별 **행 개수**입니다.  
	예: `A081 + 내담자`의 발화가 104개면 `row_count=104`.

|**`file_code_ratio_pct`**  
	해당 `file_code`가 전체 데이터(`total_rows`)에서 차지하는 **비율(%)**입니다.  
	계산: `file_code별 row_count / total_rows * 100`  
	예: `A081`이 208/1000이면 `20.8%` (같은 file_code의 상담사/내담자 행에 동일하게 붙음).

|**`speaker_ratio_pct`**  
	해당 `Speaker`가 전체 데이터에서 차지하는 **비율(%)**입니다.  
	계산: `Speaker별 row_count / total_rows * 100`  
	예: 내담자 500/1000, 상담사 500/1000이면 각각 `50.0%`.

In [6]:
# 전체 행 수
total_rows = parsed_file_name_df.height

# 원본 데이터프레임에 윈도우 함수를 사용하여 그룹별 집계 열 추가
enriched_df = parsed_file_name_df.with_columns(
    # file_code 그룹별 행 수 계산
    pl.count().over("file_code").alias("file_code_row_count"),
    # Speaker 그룹별 행 수 계산
    pl.count().over("Speaker").alias("speaker_row_count"),
)

# 위에서 계산된 그룹별 행 수를 사용하여 비율 계산
enriched_df = enriched_df.with_columns(
    # file_code별 비율
    (pl.col("file_code_row_count") / total_rows * 100)
    .round(4)
    .alias("file_code_ratio_pct"),
    # Speaker별 비율
    (pl.col("speaker_row_count") / total_rows * 100)
    .round(4)
    .alias("speaker_ratio_pct"),
)


display(enriched_df.head(20))
enriched_df.shape

/tmp/ipykernel_53817/117543745.py:7: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().over("file_code").alias("file_code_row_count"),
/tmp/ipykernel_53817/117543745.py:9: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().over("Speaker").alias("speaker_row_count"),


index,FilePath,Speaker,Content,FileName,prefix,topic,session_no,stage,file_code,file_code_row_count,speaker_row_count,file_code_ratio_pct,speaker_ratio_pct
i64,str,str,str,str,str,str,i64,str,str,u32,u32,f64,f64
207,"""gs://testprojects-453622-unzip…","""상담사""","""또 그 부분에도 참 또 그렇죠. 그래요. 일단 선생님 …","""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081""",1737,39338,2.2126,50.1089
206,"""gs://testprojects-453622-unzip…","""내담자""","""그래서 저는 막 야한 거 막 음란물 그거 즐겨 보는 경…","""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081""",1737,39159,2.2126,49.8809
205,"""gs://testprojects-453622-unzip…","""상담사""","""물론 그 범죄에 대해서 책임을 가지고 또 마땅히 또 처…","""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081""",1737,39338,2.2126,50.1089
204,"""gs://testprojects-453622-unzip…","""내담자""","""그렇죠.""","""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081""",1737,39159,2.2126,49.8809
203,"""gs://testprojects-453622-unzip…","""상담사""","""글쎄요. 저는 좀 후자적인 부분에 저도 좀 어느 정도 …","""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081""",1737,39338,2.2126,50.1089
…,…,…,…,…,…,…,…,…,…,…,…,…,…
192,"""gs://testprojects-453622-unzip…","""내담자""","""저도 생각을 그래서 어제는 오늘 아까도 집에 가면서 집…","""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081""",1737,39159,2.2126,49.8809
191,"""gs://testprojects-453622-unzip…","""상담사""","""그러니까 이거는 이거는 뭐 우리가 우연이라고 하기에는 …","""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081""",1737,39338,2.2126,50.1089
190,"""gs://testprojects-453622-unzip…","""내담자""","""아무리 아다리가 잘 맞아도 이렇게까지 잘 맞을 수가 있…","""resource_addiction_10_check_A0…","""resource""","""addiction""",10,"""check""","""A081""",1737,39159,2.2126,49.8809


(78505, 14)

## 5. 만들어진 데이터프레임 BigQuery에 올리기

In [7]:
!pip install pandas-gbq


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [9]:
# 데이터프레임을 Pandas로 변환
pandas_df = enriched_df.to_pandas()

# BigQuery에 업로드
project_id = credentials.project_id
table_id = f'{project_id}.Psychological_counseling_data.processed_data'

pandas_df.to_gbq(
    destination_table=table_id,
    project_id=project_id,
    if_exists='replace',  # 테이블이 이미 존재하면 덮어쓰기
    credentials=credentials
)

print(f"Table {table_id} uploaded successfully.")

/tmp/ipykernel_53817/281438451.py:8: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  pandas_df.to_gbq(


Table testprojects-453622.Psychological_counseling_data.processed_data uploaded successfully.
